# TinyGPT v2 — SFT Step 1: Clean Data Preparation

**Data preparation only. No model is loaded or trained in this notebook.**

This notebook starts from a clean, dependency-safe setup; inspects the actual dataset schemas; normalizes conversations; keeps OASST1 trees together in train/dev; creates response-only examples with a GPT-2 tokenizer and a hard 256-token limit; deduplicates before intentional repeats; and writes JSONL plus an audit manifest to `/kaggle/working/sft_data/`.

### Use
1. Run this notebook in Kaggle with **Internet enabled**.
2. Use **Run All from the first cell**. This data-preparation notebook does **not** require `stage4_best.pt`.
3. `handwritten_pairs.jsonl` is optional. Attach it as a Kaggle input to include handwritten pairs. If absent, other sources continue normally.
4. The four core sources are DailyDialog, Everyday Conversations, Dolly and OASST1. SODA and EmpatheticDialogues are auxiliary: the notebook records load failures and continues without them rather than crashing the whole run.
5. Inspect `manifest.json`, `sample_examples.txt`, `train.jsonl` and `dev.jsonl` in the output directory before running the separate training notebook.

Category caps are **maximums, not quotas**. The output can contain fewer than 120,000 training examples; no source is duplicated just to fill a cap. The intentional repeat settings are applied only **after** example-level deduplication and train/dev leakage removal.

In [1]:
# CELL 1 — Bootstrap, imports, configuration, tokenizer, optional handwritten input.
from pathlib import Path
from collections import Counter, defaultdict
import os, sys, json, random, re, hashlib, unicodedata, platform, time, traceback

# Import the exact APIs used later in this notebook before doing any input discovery.
try:
    import datasets
    from datasets import load_dataset
    import transformers
    from transformers import AutoTokenizer
except Exception as exc:
    raise RuntimeError(
        "Could not import datasets/transformers. In Kaggle, enable Internet and run "
        "the notebook in a standard Python notebook runtime. Original import error: " + repr(exc)
    ) from exc

CONFIG = {
    "SEED": 42,
    "DEV_FRACTION": 0.05,
    "CONTEXT_LENGTH": 256,
    "TARGET_TRAIN_EXAMPLES": 120_000,
    "MAX_EXAMPLES_PER_CONV": 4,
    "EVERYDAY_REPEAT": 2,
    "HANDWRITTEN_REPEAT": 5,  # total copies including the original
    "SODA_SCAN_LIMIT": 40_000,
    "SODA_SHUFFLE_BUFFER": 10_000,
    "ALLOW_MISSING_CORE_SOURCES": False,
    "OUTPUT_DIR": "/kaggle/working/sft_data",
    "NTFY_TOPIC": "tinygpt-siddharth-48271",
    "CATEGORY_CAPS": {
        "casual_chat": 54_000,
        "emotional_chat": 18_000,
        "short_qa": 30_000,
        "handwritten": 12_000,
    },
    "SOURCE_IDS": {
        "everyday": "HuggingFaceTB/everyday-conversations-llama3.1-2k",
        "oasst1": "OpenAssistant/oasst1",
        "dolly": "databricks/databricks-dolly-15k",
        "soda": "allenai/soda",
    },
}

random.seed(CONFIG["SEED"])
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("HF_DATASETS_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path(CONFIG["OUTPUT_DIR"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Notifications are best-effort only. A network/ntfy problem never stops preparation.
def send_ntfy(title, message):
    try:
        import urllib.request
        req = urllib.request.Request(
            f"https://ntfy.sh/{CONFIG['NTFY_TOPIC']}", data=str(message).encode("utf-8"),
            headers={"Title": str(title), "Priority": "default"}, method="POST"
        )
        with urllib.request.urlopen(req, timeout=5) as response:
            print(f"[ntfy] {title}: HTTP {response.status}")
    except Exception as exc:
        print(f"[ntfy] skipped (non-fatal): {type(exc).__name__}: {exc}")

# Do not look for or require any model checkpoint in a data-preparation notebook.
handwritten_matches = sorted(p for p in INPUT_ROOT.rglob("handwritten_pairs.jsonl") if p.is_file()) if INPUT_ROOT.exists() else []
if len(handwritten_matches) > 1:
    raise RuntimeError(
        "Found more than one handwritten_pairs.jsonl. Keep only the intended file attached:\n" +
        "\n".join(str(p) for p in handwritten_matches[:20])
    )
HANDWRITTEN_PATH = handwritten_matches[0] if handwritten_matches else None
HANDWRITTEN_RAW = []
if HANDWRITTEN_PATH is not None:
    with HANDWRITTEN_PATH.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                HANDWRITTEN_RAW.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise RuntimeError(f"Invalid JSON in {HANDWRITTEN_PATH} at line {line_no}: {exc}") from exc
    if not HANDWRITTEN_RAW:
        raise RuntimeError(f"{HANDWRITTEN_PATH} exists but contains zero non-empty JSONL records.")
    print(f"Handwritten input: {HANDWRITTEN_PATH} ({len(HANDWRITTEN_RAW):,} records)")
else:
    print("Handwritten input not attached; continuing without handwritten pairs.")

print("Python:", sys.version.split()[0], "| Platform:", platform.platform())
print("datasets:", datasets.__version__, "| transformers:", transformers.__version__)
print("Output:", OUTPUT_DIR)
print("Config:\n", json.dumps(CONFIG, indent=2))

TOKENIZER = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
if len(TOKENIZER) != 50257 or TOKENIZER.eos_token_id != 50256:
    raise RuntimeError(f"Wrong tokenizer: expected GPT-2 vocab=50257/EOS=50256; got vocab={len(TOKENIZER)}, EOS={TOKENIZER.eos_token_id}")
TOKENIZER.pad_token = TOKENIZER.eos_token
print("GPT-2 tokenizer verified: vocab=50257, EOS/PAD=50256")
send_ntfy("TinyGPT SFT data prep started", "Clean SFT data preparation started; no model training in this notebook.")

Handwritten input not attached; continuing without handwritten pairs.
Python: 3.13.15 | Platform: Linux-6.18.48+-x86_64-with-glibc2.39
datasets: 4.8.5 | transformers: 5.16.1
Output: /kaggle/working/sft_data
Config:
 {
  "SEED": 42,
  "DEV_FRACTION": 0.05,
  "CONTEXT_LENGTH": 256,
  "TARGET_TRAIN_EXAMPLES": 120000,
  "MAX_EXAMPLES_PER_CONV": 4,
  "EVERYDAY_REPEAT": 2,
  "HANDWRITTEN_REPEAT": 5,
  "SODA_SCAN_LIMIT": 40000,
  "SODA_SHUFFLE_BUFFER": 10000,
  "ALLOW_MISSING_CORE_SOURCES": false,
  "OUTPUT_DIR": "/kaggle/working/sft_data",
  "NTFY_TOPIC": "tinygpt-siddharth-48271",
  "CATEGORY_CAPS": {
    "casual_chat": 54000,
    "emotional_chat": 18000,
    "short_qa": 30000,
    "handwritten": 12000
  },
  "SOURCE_IDS": {
    "everyday": "HuggingFaceTB/everyday-conversations-llama3.1-2k",
    "oasst1": "OpenAssistant/oasst1",
    "dolly": "databricks/databricks-dolly-15k",
    "soda": "allenai/soda"
  }
}


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

GPT-2 tokenizer verified: vocab=50257, EOS/PAD=50256
[ntfy] TinyGPT SFT data prep started: HTTP 200


In [2]:
# CELL 2 — Load public datasets and print the real schema/sample before adapters are applied.
# Re-import in this cell as a deliberate guard against running it out of order.
from datasets import load_dataset

SOURCE_DATA = {}
SOURCE_META = {}
SOURCE_ERRORS = {}
SOURCE_LOAD_ATTEMPTS = defaultdict(list)
CORE_SOURCES = {"daily_dialog", "everyday", "dolly", "oasst1"}


def brief(value, limit=1400):
    text = repr(value)
    return text if len(text) <= limit else text[:limit] + " ... <truncated>"


def describe_source(name, data, dataset_id, details=None):
    SOURCE_DATA[name] = data
    if hasattr(data, "column_names"):
        columns = list(data.column_names)
        try:
            n_rows = len(data)
        except Exception:
            n_rows = None
        try:
            first = data[0] if n_rows else None
        except Exception as exc:
            first = {"sample_error": repr(exc)}
    else:
        n_rows = len(data) if hasattr(data, "__len__") else None
        first = data[0] if n_rows else None
        columns = sorted(first.keys()) if isinstance(first, dict) else []
    SOURCE_META[name] = {
        "dataset_id": dataset_id, "columns": columns, "rows": n_rows,
        "features": str(getattr(data, "features", None)), "details": details or {},
        "raw_example": first,
    }
    print("\n" + "=" * 88)
    print(f"SOURCE INSPECTION: {name} | {dataset_id}")
    print("rows:", n_rows if n_rows is not None else "stream / unknown")
    print("columns:", columns)
    print("features:", SOURCE_META[name]["features"])
    if details: print("load details:", details)
    print("raw example:", brief(first))
    return data


def attempt_load(name, method, fn):
    try:
        data = fn()
        SOURCE_LOAD_ATTEMPTS[name].append({"method": method, "result": "success"})
        return data
    except Exception as exc:
        SOURCE_LOAD_ATTEMPTS[name].append({"method": method, "error": f"{type(exc).__name__}: {exc}"})
        print(f"[{name}] {method} failed: {type(exc).__name__}: {str(exc)[:260]}")
        return None


def load_daily_dialog():
    # Parquet-first. This avoids the legacy daily_dialog.py loading script on datasets 4.x.
    candidates = [
        ("daily_dialog", "hf://datasets/daily_dialog@refs/convert/parquet/default/train/*.parquet"),
        ("li2017dailydialog/daily_dialog", "hf://datasets/li2017dailydialog/daily_dialog@refs/convert/parquet/default/train/*.parquet"),
        ("OpenRL/daily_dialog", "hf://datasets/OpenRL/daily_dialog@refs/convert/parquet/default/train/*.parquet"),
    ]
    for repo, pattern in candidates:
        data = attempt_load("daily_dialog", f"parquet:{repo}", lambda p=pattern: load_dataset("parquet", data_files=p, split="train"))
        if data is not None:
            return describe_source("daily_dialog", data, repo, {"method": "direct_parquet", "pattern": pattern})
    # Converted revision and normal mirror are last-resort paths.
    for repo in ("li2017dailydialog/daily_dialog", "roskoN/dailydialog", "OpenRL/daily_dialog"):
        data = attempt_load("daily_dialog", f"revision:{repo}", lambda r=repo: load_dataset(r, split="train", revision="refs/convert/parquet"))
        if data is not None:
            return describe_source("daily_dialog", data, repo, {"method": "converted_revision"})
        if repo != "OpenRL/daily_dialog":
            data = attempt_load("daily_dialog", f"default:{repo}", lambda r=repo: load_dataset(r, split="train"))
            if data is not None:
                return describe_source("daily_dialog", data, repo, {"method": "default_loader"})
    SOURCE_ERRORS["daily_dialog"] = SOURCE_LOAD_ATTEMPTS["daily_dialog"]
    return None


def load_everyday():
    repo = CONFIG["SOURCE_IDS"]["everyday"]
    for split in ("train_sft", "train"):
        data = attempt_load("everyday", f"split:{split}", lambda s=split: load_dataset(repo, split=s))
        if data is not None:
            return describe_source("everyday", data, repo, {"method": "direct_split", "selected_split": split})
    data = attempt_load("everyday", "dataset_dict", lambda: load_dataset(repo))
    if data is not None:
        splits = list(data.keys())
        chosen = "train_sft" if "train_sft" in splits else "train" if "train" in splits else (splits[0] if splits else None)
        if chosen is not None:
            return describe_source("everyday", data[chosen], repo, {"method": "dataset_dict", "available_splits": splits, "selected_split": chosen})
    SOURCE_ERRORS["everyday"] = SOURCE_LOAD_ATTEMPTS["everyday"]
    return None


def load_oasst():
    repo = CONFIG["SOURCE_IDS"]["oasst1"]
    data = attempt_load("oasst1", "default_train", lambda: load_dataset(repo, split="train"))
    if data is not None:
        return describe_source("oasst1", data, repo, {"method": "default_train"})
    pattern = f"hf://datasets/{repo}@refs/convert/parquet/default/train/*.parquet"
    data = attempt_load("oasst1", "direct_parquet", lambda: load_dataset("parquet", data_files=pattern, split="train"))
    if data is not None:
        return describe_source("oasst1", data, repo, {"method": "direct_parquet", "pattern": pattern})
    SOURCE_ERRORS["oasst1"] = SOURCE_LOAD_ATTEMPTS["oasst1"]
    return None

# Core source loads. We attempt each source before the final safety gate so all failures are visible.
DAILY = load_daily_dialog()
EVERYDAY = load_everyday()
DOLLY = attempt_load("dolly", "train_split", lambda: load_dataset(CONFIG["SOURCE_IDS"]["dolly"], split="train"))
if DOLLY is not None:
    describe_source("dolly", DOLLY, CONFIG["SOURCE_IDS"]["dolly"], {"method": "train_split"})
else:
    SOURCE_ERRORS["dolly"] = SOURCE_LOAD_ATTEMPTS["dolly"]
OASST = load_oasst()

# SODA is a bounded streaming scan, not a full download. It is auxiliary and never blocks the core run.
SODA = None
try:
    soda_stream = attempt_load("soda", "streaming_train", lambda: load_dataset(CONFIG["SOURCE_IDS"]["soda"], split="train", streaming=True))
    if soda_stream is not None:
        sampled = soda_stream.shuffle(seed=CONFIG["SEED"], buffer_size=CONFIG["SODA_SHUFFLE_BUFFER"]).take(CONFIG["SODA_SCAN_LIMIT"])
        SODA = list(sampled)
        describe_source("soda", SODA, CONFIG["SOURCE_IDS"]["soda"], {"method": "streamed_shuffle_sample", "scan_limit": CONFIG["SODA_SCAN_LIMIT"], "seed": CONFIG["SEED"]})
except Exception as exc:
    SOURCE_ERRORS["soda"] = repr(exc)
    print("[soda] optional source not available:", repr(exc))

# EmpatheticDialogues is auxiliary; try direct parquet first to avoid legacy script execution.
EMPATHETIC = None
for repo in ("facebook/empathetic_dialogues", "Estwld/empathetic_dialogues", "Amod/empathetic_dialogues"):
    pattern = f"hf://datasets/{repo}@refs/convert/parquet/default/train/*.parquet"
    EMPATHETIC = attempt_load("empathetic_dialogues", f"parquet:{repo}", lambda p=pattern: load_dataset("parquet", data_files=p, split="train"))
    if EMPATHETIC is not None:
        describe_source("empathetic_dialogues", EMPATHETIC, repo, {"method": "direct_parquet", "pattern": pattern})
        break
    if repo != "facebook/empathetic_dialogues":
        EMPATHETIC = attempt_load("empathetic_dialogues", f"default:{repo}", lambda r=repo: load_dataset(r, split="train"))
        if EMPATHETIC is not None:
            describe_source("empathetic_dialogues", EMPATHETIC, repo, {"method": "default_loader"})
            break
if EMPATHETIC is None:
    SOURCE_ERRORS["empathetic_dialogues"] = SOURCE_LOAD_ATTEMPTS["empathetic_dialogues"]
    print("[empathetic_dialogues] auxiliary source unavailable; continuing without it.")

missing_core = sorted(name for name, data in {
    "daily_dialog": DAILY, "everyday": EVERYDAY, "dolly": DOLLY, "oasst1": OASST
}.items() if data is None)
print("\nCore source load summary:", {k: SOURCE_META.get(k, {}).get("rows") for k in ("daily_dialog", "everyday", "dolly", "oasst1")})
if missing_core and not CONFIG["ALLOW_MISSING_CORE_SOURCES"]:
    raise RuntimeError(
        "Required core dataset(s) did not load: " + ", ".join(missing_core) +
        ". See the loader attempts printed above. Fix Internet/access or repository errors before continuing."
    )
# Validate minimum adapter schemas now, before any normalization begins.
SCHEMA_REQUIREMENTS = {
    "daily_dialog": [("dialog", "dialogue", "utterances")],
    "everyday": [("messages", "conversation", "conversations", "dialogue", "text", "prompt")],
    "dolly": [("instruction",), ("response",)],
    "oasst1": [("message_id",), ("parent_id",), ("text",), ("role",)],
}
for source_name, requirement_groups in SCHEMA_REQUIREMENTS.items():
    data = SOURCE_DATA.get(source_name)
    columns = set(SOURCE_META.get(source_name, {}).get("columns", []))
    for group in requirement_groups:
        if not any(column in columns for column in group):
            raise RuntimeError(
                f"Schema mismatch for required source {source_name!r}: columns={sorted(columns)}; "
                f"expected at least one of {group}. Review the raw example printed above."
            )
print("Schema inspection finished. Source adapters are intentionally in the next cell.")

[daily_dialog] parquet:daily_dialog failed: HfUriError: Invalid HF URI 'hf://datasets/daily_dialog@refs/convert/parquet/default/train/*.parquet'. Repository id must be 'namespace/name', got 'daily_dialog'.


default/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 3.61MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]


SOURCE INSPECTION: daily_dialog | li2017dailydialog/daily_dialog
rows: 11118
columns: ['dialog', 'act', 'emotion']
features: {'dialog': List(Value('string')), 'act': List(ClassLabel(names=['__dummy__', 'inform', 'question', 'directive', 'commissive'])), 'emotion': List(ClassLabel(names=['no emotion', 'anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise']))}
load details: {'method': 'direct_parquet', 'pattern': 'hf://datasets/li2017dailydialog/daily_dialog@refs/convert/parquet/default/train/*.parquet'}
raw example: {'dialog': ['Say , Jim , how about going for a few beers after dinner ? ', ' You know that is tempting but is really not good for our fitness . ', ' What do you mean ? It will help us to relax . ', " Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ? ", " I guess you are right.But what shall we do ? I don't feel like sitting at home . ", ' I suggest a walk over to the gym where we can play singsong and meet some of our f

...


SOURCE INSPECTION: soda | allenai/soda
rows: 40000
columns: ['PersonX', 'PersonY', 'PersonZ', 'dialogue', 'head', 'head_answer', 'literal', 'narrative', 'original_index', 'pmi_head_answer', 'pmi_relation_tail_answer', 'relation', 'relation_tail_answer', 'speakers', 'split', 'tail']
features: None
load details: {'method': 'streamed_shuffle_sample', 'scan_limit': 40000, 'seed': 42}
raw example: {'head': 'PersonX can’t stop thinking about it', 'relation': 'xAttr', 'tail': 'obsessive', 'literal': 'Maya is obsessive. Maya can’t stop thinking about it.', 'narrative': "Maya is obsessed with the color blue. She can't stop thinking about it. She loves the way it looks and feels, and she has to have it in her life.", 'dialogue': ["I'm obsessed with the color blue. I can't stop thinking about it. I love the way it looks and feels, and I have to have it in my life.", 'It sounds like you really love the color blue.', "Yes, I do. It's just so beautiful and calming. I feel like it represents everyth

default/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 5.86MB            

default/train/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]


SOURCE INSPECTION: empathetic_dialogues | facebook/empathetic_dialogues
rows: 76673
columns: ['conv_id', 'utterance_idx', 'context', 'prompt', 'speaker_idx', 'utterance', 'selfeval', 'tags']
features: {'conv_id': Value('string'), 'utterance_idx': Value('int32'), 'context': Value('string'), 'prompt': Value('string'), 'speaker_idx': Value('int32'), 'utterance': Value('string'), 'selfeval': Value('string'), 'tags': Value('string')}
load details: {'method': 'direct_parquet', 'pattern': 'hf://datasets/facebook/empathetic_dialogues@refs/convert/parquet/default/train/*.parquet'}
raw example: {'conv_id': 'hit:0_conv:1', 'utterance_idx': 1, 'context': 'sentimental', 'prompt': 'I remember going to the fireworks with my best friend. There was a lot of people_comma_ but it only felt like us in the world.', 'speaker_idx': 1, 'utterance': 'I remember going to see the fireworks with my best friend. It was the first time we ever spent time alone together. Although there was a lot of people_comma_ we 

In [3]:
# CELL 3 — Schema-aware adapters, source normalization, and 500 unique synthetic recall conversations.
# All adapters work with Hugging Face Dataset objects or lists of dicts.
from collections import defaultdict, Counter

URL_RE = re.compile(r"(?:https?://|www\.)\S+|\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b", re.I)
PHONE_RE = re.compile(r"(?<!\w)(?:\+?\d[\d .()/-]{7,}\d)(?!\w)")
CODE_RE = re.compile(r"```|<pre>|</?code>", re.I)
PROFANITY_TERMS = {"fuck", "fucking", "shit", "bitch", "cunt", "motherfucker", "nigger", "faggot"}
BAD_PREFIXES = ("as an ai", "i'm sorry, but", "im sorry, but", "sure! here")


def rows_of(data):
    if data is None:
        return []
    return data if isinstance(data, list) else (row for row in data)


def clean_text(value, daily_dialog=False):
    if value is None:
        return ""
    text = unicodedata.normalize("NFKC", str(value)).strip()
    text = text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
    text = re.sub(r"\s+", " ", text)
    if daily_dialog:
        text = text.replace(" ' t", "'t").replace(" ?", "?").replace(" !", "!").replace(" ,", ",").replace(" .", ".")
    return text.strip()


def make_conversation(cid, turns, source, category, split_group_id=None, daily=False):
    clean = []
    for item in turns or []:
        if not isinstance(item, (tuple, list)) or len(item) != 2:
            continue
        role, text = str(item[0]).lower(), clean_text(item[1], daily_dialog=daily)
        if role in ("human", "prompter", "user"):
            role = "user"
        elif role in ("gpt", "bot", "assistant"):
            role = "assistant"
        else:
            continue
        if not text:
            continue
        # Merge accidental adjacent same-role messages instead of silently swapping speakers.
        if clean and clean[-1][0] == role:
            clean[-1] = (role, clean[-1][1] + "\n" + text)
        else:
            clean.append((role, text))
    while clean and clean[0][0] != "user":
        clean.pop(0)
    while clean and clean[-1][0] != "assistant":
        clean.pop()
    if len(clean) < 2:
        return None
    if any(clean[i][0] == clean[i-1][0] for i in range(1, len(clean))):
        return None
    return {
        "conversation_id": str(cid), "split_group_id": str(split_group_id or cid),
        "source": source, "category": category, "turns": clean,
    }


def daily_conversations(data):
    out, bad = [], 0
    for i, row in enumerate(rows_of(data)):
        dialogue = row.get("dialog", row.get("dialogue", row.get("utterances"))) if isinstance(row, dict) else None
        if not isinstance(dialogue, (list, tuple)) or len(dialogue) < 2:
            bad += 1; continue
        items = list(dialogue)
        if len(items) % 2: items = items[:-1]  # drop the unmatched last user utterance
        turns = [("user" if j % 2 == 0 else "assistant", x) for j, x in enumerate(items)]
        conv = make_conversation(f"daily_dialog:{i}", turns, "daily_dialog", "casual_chat", daily=True)
        if conv: out.append(conv)
        else: bad += 1
    return out, bad


def parse_messages(value):
    if not isinstance(value, (list, tuple)):
        return None
    turns = []
    for item in value:
        if isinstance(item, str):
            turns.append(("user" if len(turns) % 2 == 0 else "assistant", item))
            continue
        if not isinstance(item, dict):
            return None
        role = str(item.get("role", item.get("from", item.get("speaker", "")))).lower()
        text = item.get("content", item.get("value", item.get("text", item.get("message", ""))))
        if role == "system":
            continue
        if role in ("user", "human", "prompter"):
            turns.append(("user", text))
        elif role in ("assistant", "gpt", "bot"):
            turns.append(("assistant", text))
        else:
            return None
    return turns


def everyday_conversations(data):
    out, bad = [], 0
    for i, row in enumerate(rows_of(data)):
        turns = None
        if not isinstance(row, dict):
            bad += 1; continue
        for key in ("messages", "conversation", "conversations", "dialogue", "text"):
            if key in row:
                turns = parse_messages(row.get(key))
                if turns:
                    break
        if not turns and isinstance(row.get("prompt"), str) and isinstance(row.get("completion"), str):
            turns = [("user", row["prompt"]), ("assistant", row["completion"])]
        conv = make_conversation(f"everyday:{i}", turns or [], "everyday", "casual_chat")
        if conv: out.append(conv)
        else: bad += 1
    return out, bad


def dolly_conversations(data):
    out, bad = [], 0
    for i, row in enumerate(rows_of(data)):
        if not isinstance(row, dict): bad += 1; continue
        user = clean_text(row.get("instruction", ""))
        context = clean_text(row.get("context", ""))
        reply = clean_text(row.get("response", ""))
        if context:
            user = user + "\n\nContext:\n" + context if user else "Context:\n" + context
        conv = make_conversation(f"dolly:{i}", [("user", user), ("assistant", reply)], "dolly", "short_qa")
        if conv: out.append(conv)
        else: bad += 1
    return out, bad


def empathetic_conversations(data):
    out, bad = [], 0
    groups = defaultdict(list)
    for row in rows_of(data):
        if isinstance(row, dict) and row.get("conv_id") is not None:
            groups[str(row["conv_id"])].append(row)
    if data is not None and not groups:
        cols = set(getattr(data, "column_names", []) or [])
        if not {"conv_id", "utterance_idx", "utterance", "speaker_idx"}.issubset(cols):
            print("[empathetic_dialogues] unsupported schema; skipping auxiliary source. Columns:", sorted(cols))
            return [], 0
    for cid, messages in groups.items():
        messages.sort(key=lambda r: int(r.get("utterance_idx", 0) or 0))
        speakers = []
        for row in messages:
            sp = str(row.get("speaker_idx", ""))
            if sp not in speakers: speakers.append(sp)
        if len(speakers) < 2:
            bad += 1; continue
        turns = []
        for row in messages:
            txt = clean_text(row.get("utterance", "")).replace("_comma_", ",")
            role = "user" if str(row.get("speaker_idx")) == speakers[0] else "assistant"
            turns.append((role, txt))
        conv = make_conversation(f"empathetic:{cid}", turns, "empathetic_dialogues", "emotional_chat")
        if conv: out.append(conv)
        else: bad += 1
    return out, bad


def soda_conversations(data):
    out, bad = [], 0
    for i, row in enumerate(rows_of(data)):
        if not isinstance(row, dict): bad += 1; continue
        dialogue, speakers = row.get("dialogue"), row.get("speakers")
        if not isinstance(dialogue, (list, tuple)) or not isinstance(speakers, (list, tuple)) or len(dialogue) != len(speakers) or len(dialogue) < 2:
            bad += 1; continue
        unique = []
        for sp in speakers:
            if sp not in unique: unique.append(sp)
        if len(unique) != 2 or any(speakers[j] == speakers[j-1] for j in range(1, len(speakers))):
            bad += 1; continue
        turns = [("user" if sp == unique[0] else "assistant", txt) for sp, txt in zip(speakers, dialogue)]
        conv = make_conversation(f"soda:{i}", turns, "soda", "casual_chat")
        if conv: out.append(conv)
        else: bad += 1
    return out, bad


def valid_oasst_context_node(row):
    return bool(row) and str(row.get("lang", "")).lower() == "en" and row.get("deleted") is False


def valid_oasst_assistant(row):
    if not valid_oasst_context_node(row) or str(row.get("role", "")).lower() != "assistant":
        return False
    if row.get("review_result") != True:
        return False
    try:
        return int(row.get("rank", -1)) == 0
    except (TypeError, ValueError):
        return False


def oasst_conversations(data):
    out, bad = [], 0
    rows = [r for r in rows_of(data) if isinstance(r, dict) and r.get("message_id") is not None]
    by_id = {str(r["message_id"]): r for r in rows}
    root_cache = {}
    def root_id(mid):
        mid = str(mid)
        if mid in root_cache: return root_cache[mid]
        cur, trail, seen = mid, [], set()
        while cur in by_id and cur not in seen:
            seen.add(cur); trail.append(cur)
            parent = by_id[cur].get("parent_id")
            if parent is None or str(parent) not in by_id:
                root = cur; break
            cur = str(parent)
        else:
            root = cur
        for node in trail: root_cache[node] = root
        return root

    for assistant in rows:
        # Iterate over assistant targets only; prompter nodes are context, never targets.
        if str(assistant.get("role", "")).lower() != "assistant":
            continue
        if not valid_oasst_assistant(assistant):
            continue
        parent = by_id.get(str(assistant.get("parent_id")))
        if not parent or str(parent.get("role", "")).lower() != "prompter" or not valid_oasst_context_node(parent):
            bad += 1; continue
        current_user, current_answer = clean_text(parent.get("text", "")), clean_text(assistant.get("text", ""))
        if not current_user or not current_answer:
            bad += 1; continue
        turns = []
        prev_assistant = by_id.get(str(parent.get("parent_id"))) if parent.get("parent_id") is not None else None
        if prev_assistant and str(prev_assistant.get("role", "")).lower() == "assistant" and valid_oasst_assistant(prev_assistant):
            prev_user = by_id.get(str(prev_assistant.get("parent_id"))) if prev_assistant.get("parent_id") is not None else None
            if prev_user and str(prev_user.get("role", "")).lower() == "prompter" and valid_oasst_context_node(prev_user):
                prev_u, prev_a = clean_text(prev_user.get("text", "")), clean_text(prev_assistant.get("text", ""))
                if prev_u and prev_a:
                    turns.extend([("user", prev_u), ("assistant", prev_a)])
        turns.extend([("user", current_user), ("assistant", current_answer)])
        tree = assistant.get("message_tree_id") or root_id(assistant["message_id"])
        cid = f"oasst1:{tree}:{assistant['message_id']}"
        group = f"oasst1-tree:{tree}"
        conv = make_conversation(cid, turns, "oasst1", "short_qa", split_group_id=group)
        if conv: out.append(conv)
        else: bad += 1
    return out, bad


def handwritten_conversations(rows):
    out, bad = [], 0
    user_keys = ("user", "prompt", "input", "question", "instruction")
    assistant_keys = ("assistant", "response", "reply", "output", "answer", "completion")
    for i, row in enumerate(rows or []):
        if not isinstance(row, dict): bad += 1; continue
        user = next((row.get(k) for k in user_keys if isinstance(row.get(k), str) and row[k].strip()), None)
        reply = next((row.get(k) for k in assistant_keys if isinstance(row.get(k), str) and row[k].strip()), None)
        if not user or not reply:
            raise RuntimeError(f"handwritten_pairs.jsonl schema mismatch at record {i}; keys={sorted(row)}")
        conv = make_conversation(f"handwritten:{i}", [("user", user), ("assistant", reply)], "handwritten", "handwritten")
        if conv: out.append(conv)
        else: bad += 1
    return out, bad

NAMES = ["Alex", "Maya", "Jordan", "Sam", "Taylor", "Riley", "Casey", "Avery", "Morgan", "Jamie", "Priya", "Arjun", "Noah", "Emma", "Leo", "Zara", "Aisha", "Daniel", "Nina", "Ishan"]
HOBBIES = ["reading", "painting", "playing chess", "cycling", "cooking", "photography", "coding", "gardening", "running", "playing guitar"]
CITIES = ["Vadodara", "Pune", "Delhi", "Mumbai", "Jaipur", "Ahmedabad", "Chennai", "Kochi", "Bengaluru", "Hyderabad"]
RECALL_QS = ["What is my name?", "Can you remind me of my name?", "What name did I tell you?", "Do you remember my name?"]
HOBBY_QS = ["What hobby did I mention?", "Which activity do I enjoy?", "What did I say I like doing?"]
CITY_QS = ["Which city did I mention?", "Where did I say I live?", "What city did I tell you about?"]


def synthetic_recall_conversations(n=500, seed=42):
    # 20 x 10 x 10 = 2,000 unique combinations; 500 are sampled deterministically.
    combos = [(name, hobby, city) for name in NAMES for hobby in HOBBIES for city in CITIES]
    rng = random.Random(seed); rng.shuffle(combos)
    out = []
    for i, (name, hobby, city) in enumerate(combos[:n]):
        if i % 3 == 0:
            question, answer = rng.choice(RECALL_QS), f"Your name is {name}."
        elif i % 3 == 1:
            question, answer = rng.choice(HOBBY_QS), f"You mentioned that you enjoy {hobby}."
        else:
            question, answer = rng.choice(CITY_QS), f"You said you live in {city}."
        turns = [
            ("user", f"My name is {name}, I enjoy {hobby}, and I live in {city}."),
            ("assistant", f"Nice to meet you, {name}. I'll remember that you enjoy {hobby} and live in {city}."),
            ("user", question), ("assistant", answer),
        ]
        conv = make_conversation(f"synthetic_recall:{i}", turns, "synthetic_recall", "handwritten")
        if conv: out.append(conv)
    contexts = [c["turns"][0][1] for c in out]
    if len(out) != n or len(set(contexts)) != n:
        raise AssertionError(f"Synthetic recall should produce {n} unique contexts; got conversations={len(out)}, unique_contexts={len(set(contexts))}")
    return out

# Build all source-specific conversation records, keeping adapter rejections visible.
SOURCE_CONVERSATIONS = {}
SOURCE_BAD_COUNTS = {}
ADAPTERS = [
    ("daily_dialog", daily_conversations, DAILY),
    ("everyday", everyday_conversations, EVERYDAY),
    ("dolly", dolly_conversations, DOLLY),
    ("oasst1", oasst_conversations, OASST),
    ("soda", soda_conversations, SODA),
    ("empathetic_dialogues", empathetic_conversations, EMPATHETIC),
]
for source_name, adapter, source_data in ADAPTERS:
    convs, rejected = adapter(source_data)
    SOURCE_CONVERSATIONS[source_name] = convs
    SOURCE_BAD_COUNTS[source_name] = rejected
    print(f"{source_name}: conversations={len(convs):,}; structurally rejected={rejected:,}; loaded={source_data is not None}")

handwritten_convs, handwritten_rejected = handwritten_conversations(HANDWRITTEN_RAW)
SOURCE_CONVERSATIONS["handwritten"] = handwritten_convs
SOURCE_BAD_COUNTS["handwritten"] = handwritten_rejected
SOURCE_CONVERSATIONS["synthetic_recall"] = synthetic_recall_conversations(500, CONFIG["SEED"])
SOURCE_BAD_COUNTS["synthetic_recall"] = 0
print("handwritten:", len(handwritten_convs), "| synthetic_recall:", len(SOURCE_CONVERSATIONS["synthetic_recall"]))
assert len(SOURCE_CONVERSATIONS["synthetic_recall"]) == 500

daily_dialog: conversations=11,118; structurally rejected=0; loaded=True
everyday: conversations=2,260; structurally rejected=0; loaded=True
dolly: conversations=15,011; structurally rejected=0; loaded=True
oasst1: conversations=7,830; structurally rejected=5; loaded=True
soda: conversations=38,709; structurally rejected=1,291; loaded=True
empathetic_dialogues: conversations=17,780; structurally rejected=64; loaded=True
handwritten: 0 | synthetic_recall: 500


In [4]:
# CELL 4 — Conversation-level train/dev split, response-only tokenization, deduplication, repeats, caps.
FILTER_COUNTS = defaultdict(Counter)
EOS_ID = TOKENIZER.eos_token_id
MAX_EXAMPLES_PER_CONV = CONFIG["MAX_EXAMPLES_PER_CONV"]


def split_conversations(conversations, dev_fraction=0.05, seed=42):
    # OASST siblings share split_group_id/message_tree_id; other data uses conversation_id.
    grouped = defaultdict(lambda: defaultdict(list))
    for conv in conversations:
        group = str(conv.get("split_group_id") or conv["conversation_id"])
        grouped[conv["source"]][group].append(conv)
    train, dev = [], []
    for source in sorted(grouped):
        ids = sorted(grouped[source])
        random.Random(seed + sum(ord(ch) for ch in source)).shuffle(ids)
        if len(ids) < 2:
            dev_ids = set()
        else:
            dev_n = max(1, int(round(len(ids) * dev_fraction)))
            dev_n = min(dev_n, len(ids) - 1)
            dev_ids = set(ids[:dev_n])
        for group_id, members in grouped[source].items():
            (dev if group_id in dev_ids else train).extend(members)
    return train, dev


def pair_hash(prompt, reply):
    def norm(text):
        text = unicodedata.normalize("NFKC", str(text)).lower()
        return re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE).strip()
    payload = re.sub(r"\s+", " ", norm(prompt)) + "\n<assistant>\n" + re.sub(r"\s+", " ", norm(reply))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def target_filter(user, reply, max_reply_tokens=60):
    if not user or not reply:
        return "empty_text"
    if len(user) > 5000 or len(reply) > 8000:
        return "extreme_text_length"
    # Drop URLs/contact data/code and a small conservative profanity list; this is not a safety classifier.
    if URL_RE.search(reply): return "assistant_url_or_email"
    if PHONE_RE.search(reply): return "assistant_phone"
    if CODE_RE.search(reply): return "assistant_code_block"
    words = set(re.findall(r"[a-zA-Z']+", reply.lower()))
    if words & PROFANITY_TERMS: return "assistant_profanity"
    if reply.lower().startswith(BAD_PREFIXES): return "boilerplate_prefix"
    reply_ids = TOKENIZER.encode(" " + reply, add_special_tokens=False)
    if len(reply_ids) + 1 > max_reply_tokens + 1: return "reply_too_long"
    # Keep >=4 supervised labels including EOS (3+ reply tokens); useful for stable token-weighted loss.
    if len(reply_ids) + 1 < 4: return "reply_too_short"
    return None


def render_prompt(turns):
    if not turns or turns[-1][0] != "user":
        raise ValueError("Prompt must end with a user turn.")
    rendered = []
    for role, text in turns:
        rendered.append(("User: " if role == "user" else "Assistant: ") + text)
    return "\n".join(rendered) + "\nAssistant:"


def build_examples(conversations, is_dev=False, seen_hashes=None):
    if seen_hashes is None: seen_hashes = set()
    examples = []
    kept_per_conversation = Counter()
    for conv in conversations:
        turns = conv["turns"]
        cid = str(conv["conversation_id"])
        # Target each assistant turn, using preceding history but always retaining the current user turn.
        for target_i in range(1, len(turns), 2):
            if kept_per_conversation[cid] >= MAX_EXAMPLES_PER_CONV:
                break
            if turns[target_i][0] != "assistant" or turns[target_i - 1][0] != "user":
                FILTER_COUNTS[conv["source"]]["nonalternating_turn"] += 1
                continue
            user, reply = turns[target_i - 1][1], turns[target_i][1]
            reason = target_filter(user, reply, max_reply_tokens=(90 if conv["category"] == "short_qa" else 60))
            if reason:
                FILTER_COUNTS[conv["source"]][reason] += 1
                continue
            history = list(turns[:target_i])  # odd count; ends with current user
            response_ids = TOKENIZER.encode(" " + reply, add_special_tokens=False) + [EOS_ID]
            if len(response_ids) < 4:
                FILTER_COUNTS[conv["source"]]["too_few_supervised_tokens"] += 1
                continue
            prompt_ids = []
            prompt = ""
            while history:
                prompt = render_prompt(history)
                prompt_ids = TOKENIZER.encode(prompt, add_special_tokens=False)
                if len(prompt_ids) >= 2 and len(prompt_ids) + len(response_ids) <= CONFIG["CONTEXT_LENGTH"]:
                    break
                # Drop only the oldest complete user/assistant pair; never truncate an utterance.
                if len(history) >= 3:
                    history = history[2:]
                else:
                    history = []
                    prompt_ids = []
                    break
            if not prompt_ids:
                FILTER_COUNTS[conv["source"]]["prompt_too_long_or_short"] += 1
                continue
            input_ids = prompt_ids + response_ids
            if len(input_ids) > CONFIG["CONTEXT_LENGTH"]:
                raise AssertionError("Internal error: encoded sequence exceeded 256 tokens")
            labels = [-100] * len(prompt_ids) + response_ids[:]
            if sum(v != -100 for v in labels) < 4 or labels[-1] != EOS_ID:
                FILTER_COUNTS[conv["source"]]["invalid_loss_mask"] += 1
                continue
            key = pair_hash(prompt, reply)
            if key in seen_hashes:
                FILTER_COUNTS[conv["source"]]["duplicate_prompt_response"] += 1
                continue
            seen_hashes.add(key)
            examples.append({
                "input_ids": input_ids, "labels": labels,
                "prompt_length": len(prompt_ids), "target_length": len(response_ids),
                "active_tokens": len(response_ids), "source": conv["source"],
                "category": conv["category"], "conversation_id": cid,
                "split_group_id": str(conv.get("split_group_id") or cid),
                "prompt_text": prompt, "target_text": reply, "is_dev": bool(is_dev),
            })
            kept_per_conversation[cid] += 1
    return examples

all_conversations = [conv for group in SOURCE_CONVERSATIONS.values() for conv in group]
if not all_conversations:
    raise RuntimeError("No usable conversations were built. Review schema/load logs before proceeding.")
train_convs, dev_convs = split_conversations(all_conversations, CONFIG["DEV_FRACTION"], CONFIG["SEED"])
print("Split conversation counts: train=", len(train_convs), "dev=", len(dev_convs))

# Dedup actual examples first. Intentional repeats must not be present yet.
train_seen_hashes = set()
base_train_examples = build_examples(train_convs, is_dev=False, seen_hashes=train_seen_hashes)
dev_seen_hashes = set()
base_dev_examples = build_examples(dev_convs, is_dev=True, seen_hashes=dev_seen_hashes)
train_pair_hashes = {pair_hash(ex["prompt_text"], ex["target_text"]) for ex in base_train_examples}
filtered_dev = []
for ex in base_dev_examples:
    if pair_hash(ex["prompt_text"], ex["target_text"]) in train_pair_hashes:
        FILTER_COUNTS[ex["source"]]["dev_train_duplicate"] += 1
    else:
        filtered_dev.append(ex)
base_dev_examples = filtered_dev

# Sanity guard against OASST message-tree leakage across splits.
train_oasst_groups = {x["split_group_id"] for x in train_convs if x["source"] == "oasst1"}
dev_oasst_groups = {x["split_group_id"] for x in dev_convs if x["source"] == "oasst1"}
assert train_oasst_groups.isdisjoint(dev_oasst_groups), "OASST1 tree leaked across train/dev split"

# Intentional repeats happen only after deduplication and leakage removal. Each repeat retains the
# original conversation ID and carries repeat_index metadata; it is not passed through dedup again.
expanded_train = []
for ex in base_train_examples:
    repeat_count = CONFIG["EVERYDAY_REPEAT"] if ex["source"] == "everyday" else CONFIG["HANDWRITTEN_REPEAT"] if ex["source"] == "handwritten" else 1
    for repeat_index in range(repeat_count):
        copy = dict(ex)
        copy["repeat_index"] = repeat_index
        expanded_train.append(copy)

rng = random.Random(CONFIG["SEED"])
rng.shuffle(expanded_train)  # shuffle before category caps to avoid source-order bias
category_counts = Counter()
train_examples = []
for ex in expanded_train:
    cat = ex["category"]
    cap = CONFIG["CATEGORY_CAPS"].get(cat, CONFIG["TARGET_TRAIN_EXAMPLES"])
    if category_counts[cat] >= cap:
        FILTER_COUNTS[ex["source"]]["category_cap"] += 1
        continue
    category_counts[cat] += 1
    train_examples.append(ex)

rng.shuffle(train_examples)
if len(train_examples) > CONFIG["TARGET_TRAIN_EXAMPLES"]:
    for ex in train_examples[CONFIG["TARGET_TRAIN_EXAMPLES"]:]:
        FILTER_COUNTS[ex["source"]]["overall_target_cap"] += 1
    train_examples = train_examples[:CONFIG["TARGET_TRAIN_EXAMPLES"]]

dev_examples = base_dev_examples
if not train_examples:
    raise RuntimeError("Zero training examples survived. Review source schemas and rejection counts.")

# Final hard checks before export.
for split_name, records in (("train", train_examples), ("dev", dev_examples)):
    for idx, ex in enumerate(records):
        assert len(ex["input_ids"]) == len(ex["labels"]), (split_name, idx, "ids/labels mismatch")
        assert len(ex["input_ids"]) <= CONFIG["CONTEXT_LENGTH"], (split_name, idx, "context too long")
        assert all(label == -100 for label in ex["labels"][:ex["prompt_length"]]), (split_name, idx, "prompt loss not masked")
        assert all(label != -100 for label in ex["labels"][ex["prompt_length"]:]), (split_name, idx, "response loss masked")
        assert ex["labels"][-1] == EOS_ID, (split_name, idx, "missing EOS supervision")
        assert sum(label != -100 for label in ex["labels"]) >= 4, (split_name, idx, "too few active labels")

print("Examples after filtering: train=", len(base_train_examples), "dev=", len(base_dev_examples))
print("Train after intentional repeats/caps:", len(train_examples))
print("Category counts:", dict(Counter(ex["category"] for ex in train_examples)))
print("Source counts:", dict(Counter(ex["source"] for ex in train_examples)))
print("Source validation counts:", dict(Counter(ex["source"] for ex in dev_examples)))
print("Filtered/rejected counts:")
for source, counts in sorted(FILTER_COUNTS.items()):
    print(" ", source, dict(counts))

Split conversation counts: train= 88542 dev= 4666


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1034 > 1024). Running this sequence through the model will result in indexing errors


Examples after filtering: train= 195443 dev= 10046
Train after intentional repeats/caps: 83666
Category counts: {'casual_chat': 54000, 'emotional_chat': 18000, 'short_qa': 10716, 'handwritten': 950}
Source counts: {'soda': 39668, 'empathetic_dialogues': 18000, 'dolly': 8766, 'oasst1': 1950, 'daily_dialog': 10131, 'everyday': 4201, 'synthetic_recall': 950}
Source validation counts: {'daily_dialog': 1356, 'dolly': 458, 'empathetic_dialogues': 1823, 'everyday': 317, 'oasst1': 102, 'soda': 5940, 'synthetic_recall': 50}
Filtered/rejected counts:
  daily_dialog {'reply_too_short': 666, 'duplicate_prompt_response': 3234, 'reply_too_long': 271, 'boilerplate_prefix': 16, 'assistant_url_or_email': 5, 'assistant_phone': 28, 'dev_train_duplicate': 365, 'category_cap': 19066}
  dolly {'reply_too_short': 464, 'prompt_too_long_or_short': 1409, 'reply_too_long': 3631, 'assistant_phone': 124, 'extreme_text_length': 113, 'assistant_url_or_email': 31, 'boilerplate_prefix': 4, 'assistant_code_block': 2, '

In [5]:
# CELL 5 — Save JSONL/manifest/sample previews and verify files on disk.
from datetime import datetime, timezone


def atomic_jsonl(path, records):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        for ex in records:
            record = {
                "input_ids": ex["input_ids"], "labels": ex["labels"],
                "prompt_length": ex["prompt_length"], "target_length": ex["target_length"],
                "active_tokens": ex["active_tokens"], "source": ex["source"],
                "category": ex["category"], "conversation_id": ex["conversation_id"],
                "split_group_id": ex["split_group_id"], "repeat_index": ex.get("repeat_index", 0),
                "prompt_text": ex["prompt_text"], "target_text": ex["target_text"],
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush(); os.fsync(f.fileno())
    tmp.replace(path)


def atomic_json(path, obj):
    path = Path(path); tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
        f.flush(); os.fsync(f.fileno())
    tmp.replace(path)


def preview_line(ex):
    # The space before target_text is intentional; target tokenization uses encode(" " + reply).
    return f"{ex['prompt_text']} {ex['target_text']}{TOKENIZER.eos_token}"

train_path = OUTPUT_DIR / "train.jsonl"
dev_path = OUTPUT_DIR / "dev.jsonl"
report_path = OUTPUT_DIR / "sample_examples.txt"
manifest_path = OUTPUT_DIR / "manifest.json"
atomic_jsonl(train_path, train_examples)
atomic_jsonl(dev_path, dev_examples)

with report_path.open("w", encoding="utf-8") as f:
    f.write("TinyGPT v2 SFT Data Preparation Report\n")
    f.write(f"Created UTC: {datetime.now(timezone.utc).isoformat()}\n")
    f.write(f"Train examples: {len(train_examples)}\nDev examples: {len(dev_examples)}\n\n")
    rng = random.Random(CONFIG["SEED"])
    for source in sorted({ex["source"] for ex in train_examples}):
        group = [ex for ex in train_examples if ex["source"] == source]
        f.write(f"=== {source} ({len(group)} training examples) ===\n")
        for idx, ex in enumerate(rng.sample(group, min(5, len(group))), 1):
            f.write(f"[{idx}] {preview_line(ex)}\n\n")

source_summary = {}
for source in sorted(set(SOURCE_CONVERSATIONS) | set(SOURCE_META) | set(SOURCE_ERRORS)):
    source_summary[source] = {
        "loaded": source in SOURCE_META,
        "dataset_id": SOURCE_META.get(source, {}).get("dataset_id"),
        "rows": SOURCE_META.get(source, {}).get("rows"),
        "valid_conversations": len(SOURCE_CONVERSATIONS.get(source, [])),
        "structural_rejections": SOURCE_BAD_COUNTS.get(source, 0),
        "load_error_or_attempts": SOURCE_ERRORS.get(source),
        "load_attempts": SOURCE_LOAD_ATTEMPTS.get(source, []),
    }

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "SFT data preparation only; no model training performed",
    "tokenizer": {"name": "gpt2", "vocab_size": len(TOKENIZER), "eos_token_id": EOS_ID},
    "config": CONFIG,
    "sources": source_summary,
    "train_count_before_repeats": len(base_train_examples),
    "dev_count_after_train_overlap_removal": len(dev_examples),
    "train_count_after_repeats_and_caps": len(train_examples),
    "train_counts_by_source": dict(Counter(ex["source"] for ex in train_examples)),
    "train_counts_by_category": dict(Counter(ex["category"] for ex in train_examples)),
    "dev_counts_by_source": dict(Counter(ex["source"] for ex in dev_examples)),
    "train_active_response_tokens": int(sum(ex["active_tokens"] for ex in train_examples)),
    "dev_active_response_tokens": int(sum(ex["active_tokens"] for ex in dev_examples)),
    "category_caps_used": dict(Counter(ex["category"] for ex in train_examples)),
    "filter_counts": {k: dict(v) for k, v in FILTER_COUNTS.items()},
    "checks": {
        "max_sequence_length_le_256": max(len(ex["input_ids"]) for ex in train_examples + dev_examples) <= CONFIG["CONTEXT_LENGTH"],
        "response_only_labels": True,
        "eos_supervised": True,
        "oasst_tree_groups_disjoint": train_oasst_groups.isdisjoint(dev_oasst_groups),
        "synthetic_recall_count": len(SOURCE_CONVERSATIONS["synthetic_recall"]),
        "synthetic_recall_contexts_unique": len({c["turns"][0][1] for c in SOURCE_CONVERSATIONS["synthetic_recall"]}) == 500,
    },
    "files": {"train": str(train_path), "dev": str(dev_path), "samples": str(report_path)},
}
atomic_json(manifest_path, manifest)

# Re-open what was written to confirm JSONL integrity and content/ID invariants.
def verify_jsonl(path, expected_count):
    n = 0
    with Path(path).open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            try: obj = json.loads(line)
            except json.JSONDecodeError as exc: raise RuntimeError(f"Output JSONL invalid: {path} line {line_no}: {exc}") from exc
            assert len(obj["input_ids"]) == len(obj["labels"])
            assert len(obj["input_ids"]) <= CONFIG["CONTEXT_LENGTH"]
            assert obj["labels"][-1] == EOS_ID
            n += 1
    assert n == expected_count, (path, n, expected_count)
    return n

print("\nOutput file verification:")
print(" train:", verify_jsonl(train_path, len(train_examples)), "records |", train_path.stat().st_size, "bytes")
print(" dev:  ", verify_jsonl(dev_path, len(dev_examples)), "records |", dev_path.stat().st_size, "bytes")
print(" manifest:", manifest_path, "|", manifest_path.stat().st_size, "bytes")
print(" samples: ", report_path, "|", report_path.stat().st_size, "bytes")
print("\nReadable preview (five per source; prompt/response boundary includes one leading space):")
rng = random.Random(CONFIG["SEED"])
for source in sorted({ex["source"] for ex in train_examples}):
    group = [ex for ex in train_examples if ex["source"] == source]
    print("\n---", source, "---")
    for idx, ex in enumerate(rng.sample(group, min(5, len(group))), 1):
        print(f"[{idx}] {preview_line(ex)}")

print("\nDone. Data prep finished; this notebook did not train a model.")
send_ntfy("TinyGPT SFT data prep completed", f"train={len(train_examples):,}; dev={len(dev_examples):,}; output={OUTPUT_DIR}")


Output file verification:
 train: 83666 records | 132655394 bytes
 dev:   10046 records | 16196288 bytes
 manifest: /kaggle/working/sft_data/manifest.json | 7022 bytes
 samples:  /kaggle/working/sft_data/sample_examples.txt | 12443 bytes

Readable preview (five per source; prompt/response boundary includes one leading space):

--- daily_dialog ---
[1] User: What's today's date?
Assistant: It's July 5th.
User: When are you going on vacation?
Assistant: I'm leaving on Sunday. We're going to Canada.
User: Really? The day after tomorrow? That's very soon.
Assistant: Yeah I know.
User: How long are you going to stay there?
Assistant: About 2 weeks.<|endoftext|>
[2] User: Hello, can I speak to Alice, please?
Assistant: Hold on, please.<|endoftext|>
[3] User: I'd like to open a one-year time deposit account and a checking account.
Assistant: Okay, if you'll please fill out these forms... ( She hands the forms to the customer. ) There's a minimum initial deposit of $ 100 for the checking acco

## Handoff to SFT training

Use `/kaggle/working/sft_data/train.jsonl`, `/kaggle/working/sft_data/dev.jsonl`, and `manifest.json` as the prepared data artifacts. Attach this notebook's Kaggle output as an input dataset to the separate training notebook if needed. Do not treat the displayed example count as a quota: category caps intentionally leave the dataset smaller when there are not enough valid unique samples.